# Detection Transformer (DETR)

Object detection has two jobs:
1. What is it ? -> classification
2. Where is it ? -> Bounding box

DETR turns object detection into a "give me fixed number of guesses, then match the guesses to real objects" problem.


YOLO : "lets look at many possible locations and predict objects from those locations".

DETR: "i`ll create 100 detectives , each detective will look at any image and try to find one object." those 100 detectives are object queries.

so
```

IMAGE
  ↓
CNN
  ↓
TRANSFORMER ENCODER
  ↓
100 OBJECT QUERIES
  ↓
TRANSFORMER DECODER
  ↓
100 PREDICTIONS
  ↓
MATCH predictions ↔ real objects

```

What decoder queries are doing ?

```
Detective #7:
    class = car
    box = [120, 80, 300, 220]
```

Why do we need CNN ?

Transformers doesnot want to directly reason about every raw pixel so we use CNN as backbone, which is visual feature extractor.

IT looks at image and gradually learns things such as:

```
early layers:
edges
lines
colors

middle layers:
eyes
wheels
ears
textures

later layers:
faces
cars
cats
etc.



so,

RAW IMAGE
    ↓
   CNN
    ↓
USEFUL VISUAL FEATURES
```

In the implementation:

```python

self.backbone=nn.Sequential(
  *list(resnet50(pretrainbed=True).children())[:-2]
)

```

means:

> Use ResNet-50  as my image feature extractor


Why the 1*1 Convolution ?

after ResNet, we might get:

```
[B, 2048, H, W]

```

But DETR`s transformers wants particular feature dimensoins called:

```
hidden_dim
```

so:
```Python
self.conv = nn.Conv2d(2048, hidden_dim, 1)

```

>  Convert my CNN features into the size that the transformer wants

if hidden_dim=256

we go from:

```
2048 channels
     ↓
256 channels
```

The 1*1 convolution is essentially being used as a channel converter.



The transformer problem:

CNN gives us:

```
[B, 256, H, W]

```

But a transformer expects a sequence.

For example:

```
[token1]
[token2]
[token3]
[token4]
...

```
it does not understand

```
height * width
```

so we flatten the image grid,

let

```
H=25
W=34
```
then

H × W = 850

so instead of:

```
256*25*34


we create:

850 tokens × 256 features

```

conceptually,

```
IMAGE FEATURE MAP

● ● ● ● ●
● ● ● ● ●
● ● ● ● ●
● ● ● ● ●

       ↓ flatten

● ● ● ● ● ● ● ● ● ● ● ● ● ...
```
Each ● represents a location in the image.


Transformers dont inherently know where things came from:

suppose we have

```
car
cat
person
```

and shuffle them:

```
person
cat
car
```

A transformer doesnot inherently know:

> "the cat was originally on the left"

so we need to tell it where every feature came from, which is purpose of **positional Embeddings**



Positional Embeddings = GPS coordinates

```
feature → "I came from x=20, y=50"

feature → "I came from x=100, y=200"

feature → "I came from x=400, y=50"

```

so the transformer receives;

```
WHAT IS HERE?
      +
WHERE IS IT?
```


Row + column Embeddings

The implementation creates:

```Python
self.row_embed=nn.Parameter(
  torch.rand(50,hidden_dim//2)
)

self.col_embed=nn.Parameter(
  torch.rand(50,hidden_dim//2)
)
```
```
ROW information
      +
COLUMN information
      ↓
LOCATION information
```

for every feature-map location:


```
(row 5, column 12)

```

We create an embedding representing that location:

then:

```

torch.cat([column_embedding, row_embedding], dim=-1)

```
combines them.



Encoder

```
CNN features
     +
position information
     ↓
Transformer Encoder
```

> lets every part of image communicate with every other relevant part

so this is where, self attention becomes useful.


Transformer can look globally

> “This region looks like an ear, and that region looks like a body, and they're probably part of the same cat.”

Encoder = looks at all image features together and builds a globally contextualized representation of the image.


DETR creates

```
100 object queries

```

```python
self.query_pos=nn.Parameter(
  torch.rand(100,hidden_dim
)

```

why 100 ?

> I will allow up to 100 objects.

They are not bounding box but learned query vectors.

During training, model learns that different queries tend to specialize in finding different objects.


Objects queries are not object location

> Object queries are learned slots tha ask the decoder to produce object predictions.


They are like:

```

"Find me an object."
"Find me another object."
"Find me another object."
...

```

The decoder figures out what those objects are and where they are.


Decoder: the detectives inspect the image:


```
ENCODER
   ↓
image understanding

and:


OBJECT QUERIES
   ↓
100 detective questions


The decoder combines them.


Image representation
        +
100 object queries
        ↓
Transformer Decoder
        ↓
100 object representations


```

Each output corresponds to one potential detection.


For example:

```
Query 1 → cat
Query 2 → couch
Query 3 → remote
Query 4 → no object
...

```

Each decoder output contain:

Each query eventually produces a vector

```
hidden_dim

```

for example:

```
256 numbers
```
That vector contains information useful for answering:


```

"What object am I looking at?"
"Where is it?"

```


Then use two predictions head.


Classification head ⁉


```python

self.linear_class=nn.Linear(
  hidden_dim,
  num_classes+1

)

```

why +1 because DeTR needs:


```
class 1
class 2
class 3
...
no object
```

The extra class  means:

> This query doesnot corresponds to an object


Bounding Box head

```python

self.linear_bbox=nn.Linear(hidden_dim,4)
```

It predicts four numbers.

Conceptually:

```
[x, y, width, height]
```

The .sigmoid() makes them approximately.

```

0 → 1

```

which allows normalized coordinates.

so:

```python
self.linear_bbox(h).sigmoid()
```

> Predict the bounding box in normalinzed image coordinates


Predictions have no order:

Suppose ground truth is [dog, cat]

but  DeTR produces: [cat,dog]

is that wrong ? ofcourse not.

The image contains exactly two same objects and order shouldnot matter, which is called:

> **Permutation invariance**

Everything is right, ordering is different. DETR solves this using **bipartite matching**



Bipartite matching = assigning detectives to objects

Imagine

```
REAL OBJECTS

Cat
Dog
Car


and

PREDICTIONS

Query 1
Query 2
Query 3
Query 4
Query 5


```

we need to determine:

Which predictions corresponds to which real objects ?

The matching algorithm might discover:

```

Query 4 → Cat
Query 1 → Dog
Query 7 → Car



The remaining queries:


Query 2 → No object
Query 3 → No object
...


```

This is Hungarian algorithm/ bipartite matching used by DETR.

Why matching is powerful ?


Because DETR can say
> " I dont care about query found the object. I only care that the one query found the correct object .

This makes the prediction a set rather than an ordered list.

Hence:

> Set - based object detection



DETR:

```
Object queries
      ↓
One-to-one matching
      ↓
Unique object predictions

```

so DETR is designed to avoid the need for traditional NMS.

> Instead of predicting many duplicate boxes and cleaning them up later, train the models to produce set of unique predictions in the first place.


| Model | Main idea |
| --- | --- |
| **DETR** | Predict a set of objects using queries |
| **Deformable DETR** | Look at fewer, useful points + multiple scales |
| **Conditional DETR** | Make queries more spatially focused |

```
DETR → Set

Deformable → Focus

Conditional → Localize

```


```
             DETR
              │
       "Find objects in image"
              │
              ▼
       ┌─────────────┐
       │     CNN     │
       │ See features│
       └──────┬──────┘
              │
              ▼
       ┌─────────────┐
       │ Positional  │
       │   "Where?"  │
       └──────┬──────┘
              │
              ▼
       ┌─────────────┐
       │   Encoder   │
       │ Understand  │
       │    image    │
       └──────┬──────┘
              │
              │
      100 object queries
              │
              ▼
       ┌─────────────┐
       │   Decoder   │
       │ Find objects│
       └──────┬──────┘
              │
              ▼
       ┌─────────────┐
       │  Prediction │
       │             │
       │ WHAT? class │
       │ WHERE? box  │
       └──────┬──────┘
              │
              ▼
       ┌─────────────┐
       │  Matching   │
       │ Who found   │
       │ which thing?│
       └──────┬──────┘
              │
              ▼
       FINAL OBJECT SET


```

“DETR uses a CNN to turn an image into useful features, a Transformer encoder to understand those features globally, and a decoder with learned object queries to produce a fixed set of object predictions. Each prediction says what the object is and where it is, and bipartite matching connects those predictions to the real objects without caring about their order.”

In [ ]:
from transformers import DetrImageProcessor, DetrForObjectDetection
import torch
from PIL import Image
import requests

# Load image
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# Load DETR
processor = DetrImageProcessor.from_pretrained(
    "facebook/detr-resnet-50"
)

model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50"
)

# Preprocess image
inputs = processor(
    images=image,
    return_tensors="pt"
)

# Run inference
outputs = model(**inputs)

# Convert predictions to image coordinates
target_sizes = torch.tensor([image.size[::-1]])

results = processor.post_process_object_detection(
    outputs,
    target_sizes=target_sizes,
    threshold=0.9
)[0]

# Print detections
for score, label, box in zip(
    results["scores"],
    results["labels"],
    results["boxes"]
):
    box = [round(i, 2) for i in box.tolist()]

    print(
        f"Detected {model.config.id2label[label.item()]} "
        f"with confidence {round(score.item(), 3)} "
        f"at location {box}"
    )


preprocessor_config.json:   0%|          | 0.00/290 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.59k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  167MB            

model.safetensors: downloading bytes:           |  0.00B            